# Week 3 — End-to-End Integration Check

**Project:** AI-Powered Task Management System
**Prepared by:** Vijayasiva

**Goal:** verify the full pipeline works as one system, using only the artifacts
already committed to `models/` and `scripts/` — no retraining:

```
raw task description
    -> Week 1 preprocessing (lowercase, punctuation removal, tokenize,
       stop-word removal, lemmatize)
    -> shared TF-IDF vectorizer
    -> SVM predicts category                              (Week 2)
    -> category + deadline/effort features
    -> XGBoost predicts priority                           (Week 3)
    -> WorkloadBalancer suggests an assignee                (Week 3)
```

This is the check called for in `docs/WEEK3_PLAN.md` / `reports/week3_progress.md`
section 3: a task goes in, a priority comes out, an assignee is suggested.

In [1]:
import sys
sys.path.insert(0, "../scripts")

import re
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from workload_balancer import WorkloadBalancer

for pkg in ["punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)

MODELS = "../models"
print("Setup complete.")

Setup complete.


## 1. Load the committed pipeline artifacts

Nothing is retrained here — every object below is exactly what Weeks 2 and 3
already produced and saved.

In [2]:
tfidf_vectorizer = joblib.load(f"{MODELS}/tfidf_vectorizer.joblib")
svm_category_model = joblib.load(f"{MODELS}/svm_tfidf.joblib")
priority_category_encoder = joblib.load(f"{MODELS}/priority_category_encoder.joblib")
priority_xgb_model = joblib.load(f"{MODELS}/priority_xgb_model.joblib")

PRIORITY_CLASSES = ["Low", "Medium", "High", "Critical"]

print("Loaded: TF-IDF vectorizer, SVM category model, category encoder, XGBoost priority model.")

Loaded: TF-IDF vectorizer, SVM category model, category encoder, XGBoost priority model.


## 2. Reusable pipeline function

Preprocessing mirrors `02_NLP_Preprocessing.ipynb` exactly (lowercase, strip
punctuation, tokenize, remove stop words, lemmatize) so a brand-new task is
treated identically to how the training data was prepared.

In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in word_tokenize(text) if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


def run_pipeline(description: str, estimated_hours: float, story_points: float,
                 days_to_deadline: float, balancer: WorkloadBalancer) -> dict:
    """Full pipeline for one new task: category -> priority -> assignee."""
    processed = preprocess_text(description)
    tfidf_vec = tfidf_vectorizer.transform([processed])

    category = svm_category_model.predict(tfidf_vec)[0]

    cat_vec = priority_category_encoder.transform(pd.DataFrame({"category": [category]}))
    num_vec = csr_matrix(np.array([[days_to_deadline, estimated_hours, story_points]]))
    priority_features = hstack([tfidf_vec, num_vec, cat_vec]).tocsr()

    priority_idx = priority_xgb_model.predict(priority_features)[0]
    priority = PRIORITY_CLASSES[priority_idx]

    suggestion = balancer.assign(priority, estimated_hours)

    return {
        "description": description,
        "predicted_category": category,
        "predicted_priority": priority,
        "suggested_assignee": suggestion.assignee_id,
        "assignee_score": round(suggestion.score, 2),
    }

print("Pipeline function ready.")

Pipeline function ready.

## 3. Build a Realistic Starting Workload State

Same approach as `08_Workload_Balancing.ipynb`: derive each assignee's current
experience and open-task count from the dataset's own history, so the demo
below reflects a plausible "right now" rather than everyone starting idle.

In [4]:
df = pd.read_csv("../data/processed/tasks_nlp.csv")
df["created_date"] = pd.to_datetime(df["created_date"])
df = df.sort_values("created_date").reset_index(drop=True)

exp_by_user = df.groupby("assignee_id")["assignee_experience_years"].last()
open_by_user = df.groupby("assignee_id")["assignee_open_tasks"].last()
all_users = sorted(df["assignee_id"].unique())

users_df = pd.DataFrame({
    "assignee_id": all_users,
    "experience_years": [exp_by_user[u] for u in all_users],
    "open_tasks": [int(open_by_user[u]) for u in all_users],
})
balancer = WorkloadBalancer.from_dataframe(users_df)
print(f"Workload state initialized for {len(all_users)} assignees.")

Workload state initialized for 25 assignees.


## 4. Run New, Unseen Tasks Through the Full Pipeline

None of these exact sentences appear in the training data — this checks the
pipeline generalizes, not that it memorized.

In [5]:
example_tasks = [
    dict(description="URGENT: payment gateway crashes on checkout for all users!!",
        estimated_hours=6.0, story_points=5, days_to_deadline=1),
    dict(description="Write the onboarding documentation for the new billing system",
        estimated_hours=3.0, story_points=2, days_to_deadline=14),
    dict(description="Add dark mode support to the admin panel settings page",
        estimated_hours=10.0, story_points=8, days_to_deadline=21),
    dict(description="Investigate why the notification service is dropping messages",
        estimated_hours=5.0, story_points=3, days_to_deadline=3),
    dict(description="Set up automated regression tests for the checkout flow",
        estimated_hours=8.0, story_points=5, days_to_deadline=10),
]

results = [run_pipeline(balancer=balancer, **t) for t in example_tasks]
results_df = pd.DataFrame(results)
results_df

,description,predicted_category,predicted_priority,suggested_assignee,assignee_score
0,URGENT: payment gateway crashes on checkout fo...,Bug Fix,Critical,USER-23,-3.25
1,Write the onboarding documentation for the new...,Documentation,Low,USER-16,-1.05
2,Add dark mode support to the admin panel setti...,Feature,Low,USER-16,-2.83
3,Investigate why the notification service is dr...,Bug Fix,High,USER-12,-3.30
4,Set up automated regression tests for the chec...,Testing,Medium,USER-05,-3.53


## 5. Verification

The pipeline is confirmed working end-to-end if, for every example above:
- a `predicted_category` was produced (SVM ran successfully on new text)
- a `predicted_priority` was produced (XGBoost ran successfully on the assembled features)
- a `suggested_assignee` was produced (the balancer scored all 25 candidates and returned one)
- urgent/short-deadline tasks receive higher priority than routine ones (sanity check on direction, not exact match)

In [6]:
checks = {
    "All tasks got a category": results_df["predicted_category"].notna().all(),
    "All tasks got a priority": results_df["predicted_priority"].isin(PRIORITY_CLASSES).all(),
    "All tasks got a suggested assignee": results_df["suggested_assignee"].isin(all_users).all(),
    "Urgent 1-day task ranks Medium+": results_df.iloc[0]["predicted_priority"] in ["Medium", "High", "Critical"],
    "Distinct assignees suggested (balancer isn't picking one person)":
        results_df["suggested_assignee"].nunique() >= 2,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {name}")

assert all(checks.values()), "Integration check failed - see above"
print("\nEnd-to-end pipeline verified: description -> category -> priority -> assignee.")

PASS - All tasks got a category
PASS - All tasks got a priority
PASS - All tasks got a suggested assignee
PASS - Urgent 1-day task ranks Medium+
PASS - Distinct assignees suggested (balancer isn't picking one person)

End-to-end pipeline verified: description -> category -> priority -> assignee.


## 6. Conclusion

The full pipeline runs end-to-end on unseen task descriptions using only the
artifacts already committed in Weeks 2–3, with no retraining required:

1. **Preprocessing** (Week 1) normalizes new text identically to the training data
2. **SVM on TF-IDF** (Week 2) classifies the task's category
3. **XGBoost** (Week 3) predicts priority from text + category + deadline/effort
4. **WorkloadBalancer** (Week 3) suggests an assignee, balancing experience fit against current load

This satisfies the Week 3 integration item in `docs/WEEK3_PLAN.md`. The pipeline
function `run_pipeline()` above is the basis for the Week 4 dashboard/demo.